# Colab ComfyUI(简版)
在 Colab 上直接运行一个标准 ComfyUI,在浏览器里完成全部出图/重绘流程。不需要远程插件、不需要本机配置。

- 模型读取 `MyDrive/ComfyUI/models/`(checkpoints / loras / vae / clip / text_encoders / diffusion_models)
- 输入图读取 `MyDrive/ComfyUI/input/`,输出直接写到 `MyDrive/ComfyUI/output/`(会话结束也不丢)
- 空闲 30 分钟自动释放 GPU

从上往下依次运行。第 5 格会打印访问地址。

## 1. GPU 与 Drive

In [ ]:
import os, sys, subprocess, json, re, time, threading
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_COMFY = Path('/content/drive/MyDrive/ComfyUI')
MODELS_DIR = DRIVE_COMFY / 'models'
INPUT_DIR  = DRIVE_COMFY / 'input'
OUTPUT_DIR = DRIVE_COMFY / 'output'
for d in (INPUT_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert MODELS_DIR.is_dir(), 'MyDrive/ComfyUI/models not found'
print({p.name: len(list(p.iterdir())) for p in MODELS_DIR.iterdir() if p.is_dir()})

## 2. 安装 ComfyUI(最新版)和 ComfyUI-Manager
需要额外自定义节点(SAM3、Impact Pack 等)时,在网页里用 Manager 安装即可;自定义节点存在 Colab 本地盘,新会话需要重新装。

In [ ]:
COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')], check=True)
manager = COMFY / 'custom_nodes' / 'ComfyUI-Manager'
if not manager.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI-Manager.git', str(manager)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(manager / 'requirements.txt')], check=False)
print('ready')

## 3. 模型路径指向 Drive

In [ ]:
yaml = f'''
drive:
  base_path: {MODELS_DIR}
  checkpoints: checkpoints
  loras: loras
  vae: vae
  clip: clip
  text_encoders: text_encoders
  diffusion_models: diffusion_models
'''
EXTRA = Path('/content/extra_model_paths.yaml')
EXTRA.write_text(yaml)
print(yaml)

## 3b. 安装重绘节点 ComfyUI-SEGSInpaint
替代你原工作流里的 `SAM3BoxesToSEGS` 和 `RemoteSEGSDetailer`,全部在本机(Colab)执行,不需要插件、URL、token。类名和参数名与原工作流一致,所以重绘出来的 PNG 可以直接拖进网页重新运行。
检查点从 `checkpoints/` 读取(如 `xaviernocturne_v3.safetensors`),LoRA 从 `loras/` 读取。
每个文件一格,必须在启动 ComfyUI(第 4 步)之前运行。

In [ ]:
os.makedirs('/content/ComfyUI/custom_nodes/ComfyUI-SEGSInpaint', exist_ok=True)

In [ ]:
%%writefile /content/ComfyUI/custom_nodes/ComfyUI-SEGSInpaint/geometry.py
"""Pure-python box/crop math (no torch, easy to unit-test)."""


def boxes_from_input(bboxes, img_w, img_h):
    """Accepts CreateBoundingBoxes output ([[{x,y,width,height}, ...]]) or a flat list of dicts.
    Returns a list of integer (x1, y1, x2, y2) clipped to the image; empty boxes are dropped."""
    flat = []

    def walk(item):
        if isinstance(item, dict):
            flat.append(item)
        elif isinstance(item, (list, tuple)):
            for sub in item:
                walk(sub)

    walk(bboxes)
    out = []
    for b in flat:
        if not all(k in b for k in ("x", "y", "width", "height")):
            continue
        x1 = max(0, min(img_w, int(round(b["x"]))))
        y1 = max(0, min(img_h, int(round(b["y"]))))
        x2 = max(0, min(img_w, int(round(b["x"] + b["width"]))))
        y2 = max(0, min(img_h, int(round(b["y"] + b["height"]))))
        if x2 - x1 >= 2 and y2 - y1 >= 2:
            out.append((x1, y1, x2, y2))
    return out


def expand_box(box, padding, img_w, img_h):
    x1, y1, x2, y2 = box
    return (max(0, x1 - padding), max(0, y1 - padding), min(img_w, x2 + padding), min(img_h, y2 + padding))


def crop_region_for(box, crop_factor, img_w, img_h, min_size=64):
    """Scale `box` about its centre by crop_factor and clip to the image."""
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    w = max((x2 - x1) * crop_factor, min_size)
    h = max((y2 - y1) * crop_factor, min_size)
    rx1 = max(0, int(round(cx - w / 2.0)))
    ry1 = max(0, int(round(cy - h / 2.0)))
    rx2 = min(img_w, int(round(cx + w / 2.0)))
    ry2 = min(img_h, int(round(cy + h / 2.0)))
    return (rx1, ry1, rx2, ry2)


def target_size(ref_w, ref_h, crop_w, crop_h, guide_size, max_size, multiple=8):
    """Impact-Pack style: scale so the shorter reference side reaches guide_size, capped by max_size.
    Returns (new_w, new_h, upscale)."""
    upscale = guide_size / max(1, min(ref_w, ref_h))
    new_w, new_h = crop_w * upscale, crop_h * upscale
    longest = max(new_w, new_h)
    if longest > max_size:
        upscale *= max_size / longest
        new_w, new_h = crop_w * upscale, crop_h * upscale
    snap = lambda v: max(multiple, int(round(v / multiple)) * multiple)
    return snap(new_w), snap(new_h), upscale


In [ ]:
%%writefile /content/ComfyUI/custom_nodes/ComfyUI-SEGSInpaint/__init__.py
"""ComfyUI-SEGSInpaint

Standalone (no Impact Pack, no remote plugin) replacements for the two nodes used by the
supervised-inpaint workflows:

  * SAM3BoxesToSEGS     - drawn boxes -> SEGS (the box itself is the mask, no SAM needed)
  * RemoteSEGSDetailer  - per-SEGS crop -> upscale to guide_size -> masked img2img -> paste back

Class names and input names are kept identical to the original workflows so the API-format
workflow embedded in the generated PNGs can be loaded and run unchanged. `url` and `token`
are accepted for compatibility and ignored: everything runs on this ComfyUI instance.
"""
import json
import os
import sys

import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(__file__))
from geometry import boxes_from_input, expand_box, crop_region_for, target_size  # noqa: E402

import comfy.samplers
import comfy.sd
import comfy.utils
import folder_paths
import nodes as comfy_nodes


# ----------------------------------------------------------------------------- helpers
def gaussian_blur_mask(mask, radius):
    """mask: [1,1,H,W] float. Separable gaussian, radius in pixels (0 = no-op)."""
    radius = int(radius)
    if radius <= 0:
        return mask
    sigma = max(radius / 2.0, 0.5)
    k = torch.arange(-radius, radius + 1, dtype=mask.dtype, device=mask.device)
    k = torch.exp(-(k ** 2) / (2 * sigma ** 2))
    k = k / k.sum()
    pad = radius
    x = F.pad(mask, (pad, pad, pad, pad), mode="replicate")
    x = F.conv2d(x, k.view(1, 1, 1, -1))
    x = F.conv2d(x, k.view(1, 1, -1, 1))
    return x.clamp(0, 1)


_MODEL_CACHE = {}


def load_base_model(model_name):
    if model_name not in _MODEL_CACHE:
        _MODEL_CACHE.clear()  # keep a single checkpoint resident
        path = folder_paths.get_full_path_or_raise("checkpoints", model_name)
        out = comfy.sd.load_checkpoint_guess_config(
            path, output_vae=False, output_clip=False,
            embedding_directory=folder_paths.get_folder_paths("embeddings"))
        _MODEL_CACHE[model_name] = out[0]
    return _MODEL_CACHE[model_name]


def apply_loras(model, loras_json):
    try:
        entries = json.loads(loras_json) if loras_json and loras_json.strip() else []
    except json.JSONDecodeError as e:
        raise ValueError(f"`loras` is not valid JSON: {e}")
    for entry in entries:
        strength = float(entry.get("strength", 0))
        if not entry.get("name") or strength == 0:
            continue
        path = folder_paths.get_full_path_or_raise("loras", entry["name"])
        lora = comfy.utils.load_torch_file(path, safe_load=True)
        model, _ = comfy.sd.load_lora_for_models(model, None, lora, strength, 0)
    return model


# ----------------------------------------------------------------------------- nodes
class SAM3BoxesToSEGS:
    """Boxes -> SEGS. `padding` grows each drawn box (the mask); `crop_factor` scales that
    padded box about its centre to get the context crop that is actually inpainted."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "bboxes": ("BOUNDING_BOX",),
            "crop_factor": ("FLOAT", {"default": 2.0, "min": 1.0, "max": 10.0, "step": 0.1}),
            "padding": ("INT", {"default": 16, "min": 0, "max": 512, "step": 1}),
            "image": ("IMAGE",),
        }}

    RETURN_TYPES = ("SEGS",)
    FUNCTION = "convert"
    CATEGORY = "image/detailer"

    def convert(self, bboxes, crop_factor, padding, image):
        _, h, w, _ = image.shape
        segs = []
        for box in boxes_from_input(bboxes, w, h):
            mask_box = expand_box(box, padding, w, h)
            region = crop_region_for(mask_box, crop_factor, w, h)
            rx1, ry1, rx2, ry2 = region
            mask = torch.zeros((ry2 - ry1, rx2 - rx1), dtype=torch.float32)
            mask[mask_box[1] - ry1:mask_box[3] - ry1, mask_box[0] - rx1:mask_box[2] - rx1] = 1.0
            segs.append({"bbox": mask_box, "crop_region": region, "mask": mask})
        if not segs:
            print("[SEGSInpaint] SAM3BoxesToSEGS: no valid boxes, detailer will be a no-op")
        return ({"shape": (h, w), "segs": segs},)


class RemoteSEGSDetailer:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "guide_size": ("FLOAT", {"default": 512, "min": 64, "max": 4096, "step": 8}),
            "guide_size_for": ("BOOLEAN", {"default": True, "label_on": "bbox", "label_off": "crop_region"}),
            "max_size": ("FLOAT", {"default": 1024, "min": 64, "max": 4096, "step": 8}),
            "seed": ("INT", {"default": 0, "min": 0, "max": 0xffffffffffffffff}),
            "steps": ("INT", {"default": 20, "min": 1, "max": 10000}),
            "cfg": ("FLOAT", {"default": 8.0, "min": 0.0, "max": 100.0, "step": 0.1}),
            "sampler_name": (comfy.samplers.KSampler.SAMPLERS,),
            "scheduler": (comfy.samplers.KSampler.SCHEDULERS,),
            "denoise": ("FLOAT", {"default": 0.5, "min": 0.0001, "max": 1.0, "step": 0.01}),
            "feather": ("INT", {"default": 5, "min": 0, "max": 100}),
            "noise_mask": ("BOOLEAN", {"default": True}),
            "force_inpaint": ("BOOLEAN", {"default": True}),
            "cycle": ("INT", {"default": 1, "min": 1, "max": 10}),
            "noise_mask_feather": ("INT", {"default": 0, "min": 0, "max": 100}),
            "tiled_encode": ("BOOLEAN", {"default": False}),
            "tiled_decode": ("BOOLEAN", {"default": False}),
            "model_name": (folder_paths.get_filename_list("checkpoints"),),
            "loras": ("STRING", {"default": "[]", "multiline": True}),
            "image": ("IMAGE",),
            "segs": ("SEGS",),
            "vae": ("VAE",),
            "positive": ("CONDITIONING",),
            "negative": ("CONDITIONING",),
        }, "optional": {
            # accepted for workflow compatibility, ignored (everything runs locally)
            "url": ("STRING", {"default": ""}),
            "token": ("STRING", {"default": ""}),
        }}

    RETURN_TYPES = ("IMAGE",)
    FUNCTION = "detail"
    CATEGORY = "image/detailer"

    # -- core: one crop through VAE-encode / sample / VAE-decode -------------------------
    def _inpaint_crop(self, crop, mask, new_w, new_h, model, vae, positive, negative, seed, steps,
                      cfg, sampler_name, scheduler, denoise, use_noise_mask, noise_mask_feather,
                      tiled_encode, tiled_decode, cycle, upscale):
        pix = F.interpolate(crop.permute(0, 3, 1, 2), size=(new_h, new_w), mode="bicubic",
                            align_corners=False).clamp(0, 1).permute(0, 2, 3, 1)[..., :3]
        m = F.interpolate(mask[None, None], size=(new_h, new_w), mode="bilinear", align_corners=False)
        m = gaussian_blur_mask(m, round(noise_mask_feather * upscale))
        for _ in range(cycle):
            latent = vae.encode_tiled(pix) if tiled_encode else vae.encode(pix)
            lat = {"samples": latent}
            if use_noise_mask:
                lat["noise_mask"] = m
            out = comfy_nodes.common_ksampler(model, seed, steps, cfg, sampler_name, scheduler,
                                              positive, negative, lat, denoise=denoise)[0]
            samples = out["samples"]
            pix = vae.decode_tiled(samples) if tiled_decode else vae.decode(samples)
            if pix.dim() == 5:
                pix = pix.reshape(-1, *pix.shape[-3:])
            pix = pix[..., :3].clamp(0, 1)
        return pix

    def detail(self, guide_size, guide_size_for, max_size, seed, steps, cfg, sampler_name, scheduler,
               denoise, feather, noise_mask, force_inpaint, cycle, noise_mask_feather, tiled_encode,
               tiled_decode, model_name, loras, image, segs, vae, positive, negative, url="", token=""):
        seg_list = segs["segs"]
        if not seg_list:
            return (image,)
        model = apply_loras(load_base_model(model_name), loras)
        result = image.clone()
        for b in range(result.shape[0]):
            for i, seg in enumerate(seg_list):
                rx1, ry1, rx2, ry2 = seg["crop_region"]
                bx1, by1, bx2, by2 = seg["bbox"]
                crop = result[b:b + 1, ry1:ry2, rx1:rx2, :]
                ch, cw = crop.shape[1], crop.shape[2]
                ref_w, ref_h = (bx2 - bx1, by2 - by1) if guide_size_for else (cw, ch)
                new_w, new_h, upscale = target_size(ref_w, ref_h, cw, ch, guide_size, max_size)
                if upscale <= 1.0 and not force_inpaint:
                    print(f"[SEGSInpaint] seg {i}: already >= guide_size, skipped")
                    continue
                mask = seg["mask"].to(crop.device)
                new_pix = self._inpaint_crop(crop, mask, new_w, new_h, model, vae, positive, negative,
                                             seed, steps, cfg, sampler_name, scheduler, denoise,
                                             noise_mask, noise_mask_feather, tiled_encode, tiled_decode,
                                             cycle, upscale)
                new_pix = F.interpolate(new_pix.permute(0, 3, 1, 2), size=(ch, cw), mode="bicubic",
                                        align_corners=False).clamp(0, 1).permute(0, 2, 3, 1)
                blend = gaussian_blur_mask(mask[None, None], feather)[0, 0][None, :, :, None]
                result[b:b + 1, ry1:ry2, rx1:rx2, :] = crop * (1 - blend) + new_pix.to(crop.dtype) * blend
        return (result,)


NODE_CLASS_MAPPINGS = {"SAM3BoxesToSEGS": SAM3BoxesToSEGS, "RemoteSEGSDetailer": RemoteSEGSDetailer}
NODE_DISPLAY_NAME_MAPPINGS = {
    "SAM3BoxesToSEGS": "Boxes → SEGS (no SAM)",
    "RemoteSEGSDetailer": "SEGS Detailer (local, drop-in)",
}


## 4. 启动 ComfyUI

In [ ]:
import requests
if 'comfy_proc' in globals() and comfy_proc.poll() is None:
    raise RuntimeError('ComfyUI is already running.')
comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen([
    sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
    '--extra-model-paths-config', str(EXTRA),
    '--input-directory', str(INPUT_DIR), '--output-directory', str(OUTPUT_DIR),
    '--preview-method', 'auto',
], cwd=COMFY, stdout=comfy_log, stderr=subprocess.STDOUT)
for _ in range(120):
    if comfy_proc.poll() is not None:
        print(Path('/content/comfyui.log').read_text()[-5000:])
        raise RuntimeError('ComfyUI exited; see log above.')
    try:
        if requests.get('http://127.0.0.1:8188/system_stats', timeout=3).status_code == 200:
            print('ComfyUI is up on port 8188')
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError('ComfyUI did not start; check /content/comfyui.log')

## 5. 打开网页
**注意**:ComfyUI 自身没有登录。下面的 Cloudflare 地址是随机的公开链接,**任何拿到链接的人都能用你的 GPU 并通过 Manager 装节点**,不要分享、不要截图带出地址。
想更安全可以把 `TUNNEL` 改成 `'colab'`,走 Colab 自带的代理(需要登录你的 Google 账号),但个别情况下 WebSocket 进度可能不稳定。

In [ ]:
TUNNEL = 'cloudflared'   # 'cloudflared' 或 'colab'

if TUNNEL == 'colab':
    from google.colab import output
    output.serve_kernel_port_as_window(8188, path='/')
else:
    cf = Path('/content/cloudflared')
    if not cf.exists():
        url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
        with requests.get(url, stream=True, timeout=(30, 120)) as r:
            r.raise_for_status()
            with cf.open('wb') as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        cf.chmod(0o755)
    if 'tunnel' not in globals() or tunnel.poll() is not None:
        tunnel = subprocess.Popen([str(cf), 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
                                  stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/tunnel.log').read_text())
        if m:
            print('ComfyUI:', m.group(0))
            break
        time.sleep(1)
    else:
        raise TimeoutError('Tunnel URL not ready; see /content/tunnel.log')

## 6. 空闲自动释放 GPU
队列里有任务、或有新任务完成,都会重置计时。只在网页里改工作流不算活动。修改 `IDLE_MINUTES` 后重新运行此格。

In [ ]:
from google.colab import runtime
IDLE_MINUTES = 30
def _activity():
    q = requests.get('http://127.0.0.1:8188/queue', timeout=5).json()
    h = requests.get('http://127.0.0.1:8188/history?max_items=1', timeout=5).json()
    return bool(q.get('queue_running') or q.get('queue_pending')), (next(iter(h), None))

def _watch(stop, limit):
    last_active, last_hist = time.monotonic(), None
    while not stop.wait(10):
        try:
            busy, hist = _activity()
        except requests.RequestException:
            last_active = time.monotonic()      # cannot confirm idle
            continue
        if busy or hist != last_hist:
            last_active = time.monotonic()
        last_hist = hist
        if time.monotonic() - last_active >= limit:
            print('Idle for', IDLE_MINUTES, 'minutes; releasing runtime.')
            runtime.unassign()
            return

if 'idle_stop' in globals():
    idle_stop.set(); time.sleep(11)
idle_stop = threading.Event()
threading.Thread(target=_watch, args=(idle_stop, IDLE_MINUTES * 60), daemon=True).start()
print('Auto release enabled:', IDLE_MINUTES, 'min idle')

## 手动释放 / 关闭定时器 / 看日志
```python
from google.colab import runtime; runtime.unassign()   # 立即释放
idle_stop.set()                                         # 只关定时器
!tail -n 80 /content/comfyui.log                        # ComfyUI 日志
```